# Hafta 12 — gerçek veride GPT-2 ve HellaSwag

**Hafta 11 sonuç ZIP'ini önce indirin. Mevcut L4 oturumunu koruyabilirsiniz.**
Defteri yükledikten sonra Colab'ın **Düzenle → Tüm hücreleri seç → Hücreleri kopyala**
seçenekleriyle bu hücreleri açık hafta 11 defterinin sonuna yapıştırabilirsiniz.
Alternatif olarak bu defteri kendi GPU oturumunda çalıştırabilirsiniz; yeni defterin
aynı sanal makineyi otomatik paylaştığını varsaymayın.

Akış: kaynaklar → GPU → testler → Drive yedeği → 20M token hazırlığı →
HellaSwag verisini hazırla → yaklaşık 60 dakika eğitim → kendi modelin → OpenAI GPT-2 → sonuç ZIP'i.
Veri hazırlığı ve HellaSwag ölçümleri 60 dakikalık eğitim bütçesine dahil değildir.
İlk derleme, periyodik val/örnekleme/checkpoint süreleri eğitim duvar bütçesine dahildir;
son değerlendirme/kayıt süreleri bütçeyi biraz aşabilir.

124M model sıfırdan başlar. Hafta 11'in kısa demo metninde ezberlenen ağırlıklarını kullanmaz.


## 1. Kaynakları doğrula ve bağımlılıkları kur

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

# Prefer running these cells in the CURRENT Week 11 notebook/runtime.
ROOT = Path('/content/neuron-to-gpt-week12')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_HASHES = {'week10/model.py': 'c3880320d61666d7c79f861dba74fb242ad09085fb7e03dd45ea64dde55f9535', 'week11/training.py': '4f2726530f52a6b2a4ba42afde9d5a2b1848445aa94c315f2c7d3aba2051fe05', 'week12/requirements.txt': 'fbaa3a25f33137be3bf04bf3bff20eb9555bc54fe0d9bdf5f038fdb0ba664096', 'week12/common.py': 'eef5d129baa5ee535e3f73712fef517178883def84119a6b3f8667949cd465db', 'week12/data.py': '1cea648faf5728b394b1928db9da29a2f11645f180d46c82468c7400fd288354', 'week12/evaluate.py': '5ccd1e957f189f6ba3d783825e42d3cb21d53f315ea40e8a8fcd140f7d0f5867', 'week12/make_colab_notebook.py': 'a2a267d41214cc450e09e26175722c7a360eacfefdb29d807195c0ba17057a5d', 'week12/prepare_data.py': '23aff9e61c8ebace9f756a6f076d0f59460348963781f560881a7776cb0e4094', 'week12/report.py': '5533dbc4e0e2ec1bce3657dc7cc647556d14bc9bf1c2992dfe46c30445d2fc7c', 'week12/test_pipeline.py': 'ac5e975febb3379eed61e35a958116b9d4c3bfacb851d85992a367e273364f13', 'week12/train.py': '53b60b9c59560ed897c57e06534f8baa495497587e93fbb004ba0b1adbfb1dc2'}
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'e5bf8606cb224bd578fa98f00718ae57ae465d4662fa9ceb005d94365285e655'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == set(SOURCE_HASHES)
    for filename, expected in SOURCE_HASHES.items():
        raw = archive.read(filename)
        assert hashlib.sha256(raw).hexdigest() == expected
        destination = ROOT / filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(raw)
os.environ['HF_HOME'] = str(ROOT / 'week12/.cache/huggingface')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
# Reuse the tokenizer cache if Week 11 ran in this runtime.
old_cache = Path('/content/neuron-to-gpt-week11/week11/.cache/tiktoken')
os.environ['TIKTOKEN_CACHE_DIR'] = str(old_cache if old_cache.exists() else ROOT / 'week10/.cache/tiktoken')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'week12/requirements.txt')], check=True)
print('Week 12 sources verified:', json.dumps(SOURCE_HASHES, indent=2))


## 2. Bağlı GPU ve ayarlar
L4 için başlangıç: B=4, T=256, accumulation=4. A100 aramak için mevcut oturumu silmek gerekmiyor.

In [ ]:
import re, torch
assert torch.cuda.is_available(), 'GPU runtime required. Keep the current L4 if it is connected.'
GPU_NAME = torch.cuda.get_device_name(0)
RUN_NAME = re.sub(r'[^a-z0-9]+', '_', GPU_NAME.lower()).strip('_')
DATA = ROOT / 'week12/data/fineweb20m'
RESULTS = ROOT / 'week12/outputs' / RUN_NAME
RESULTS.mkdir(parents=True, exist_ok=True)

TRAIN_TOKENS, VAL_TOKENS = 20_000_000, 131_072
MINUTES, BATCH_SIZE, SEQUENCE_LENGTH, ACCUMULATION = 60, 4, 256, 4
COMPILE = True
HELLASWAG_EXAMPLES = 1000
BACKUP = None  # Set by the optional Drive cell below.
print('GPU:', GPU_NAME, '| torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Precision:', 'BF16' if torch.cuda.is_bf16_supported() else 'FP16 + GradScaler')
print('Effective batch tokens:', BATCH_SIZE * SEQUENCE_LENGTH * ACCUMULATION)
subprocess.run(['nvidia-smi'], check=True)

def execute(arguments):
    command = [sys.executable, '-u', *arguments]
    # Notebook output streams do not always expose inherited child FDs.
    # Read the pipe and print in the notebook process so progress is visible.
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True,
                               encoding='utf-8', errors='replace', bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()


## 3. Doğruluk testleri
Shard sınırları, validation ayrımı, HellaSwag maskesi ve uzunluk normalizasyonu.

In [ ]:
execute(['-m', 'unittest', 'discover', '-s', 'week12', '-p', 'test_*.py'])


## 4. Checkpoint için Google Drive yedeği (önerilir)
Her 1.000 güncellemede ve eğitim sonunda `last.pt` ile metrikler yedeklenir. Resume checkpoint'i yaklaşık 1,5 GB'dir. Drive kullanmak istemiyorsanız bu hücreyi atlayabilirsiniz; yerel `/content` dosyaları oturum silinince kaybolur.

In [ ]:
# Recommended: preserve the latest checkpoint in your own Google Drive.
# This cell opens Google's normal authorization dialog.
from google.colab import drive
drive.mount('/content/drive')
BACKUP = Path('/content/drive/MyDrive/YZ50/week12') / RUN_NAME
BACKUP.mkdir(parents=True, exist_ok=True)
print('Periodic checkpoint backup:', BACKUP)


## 5. FineWeb-Edu: 20M train + ayrı validation
Veri kümesinin sabit sürümünden streaming yapılır. GPT-2 tokenizer, 1M token'lık uint16 NPY shard'ları; validation için ayrı belgeler. İlk bağlantı/Parquet okuması zaman alabilir. Tamamlandığında token, shard ve belge sayıları görünür.

In [ ]:
execute([
    'week12/prepare_data.py', '--output-dir', str(DATA),
    '--train-tokens', str(TRAIN_TOKENS), '--val-tokens', str(VAL_TOKENS),
])
manifest = json.loads((DATA / 'manifest.json').read_text())
print({split: {'tokens': row['tokens'], 'shards': len(row['shards']), 'documents': row['documents']}
       for split, row in manifest['splits'].items()})


## 6. HellaSwag'in ilk 1.000 validation örneğini hazırla
Veri erişimini uzun eğitimden önce kontrol ediyoruz. Eski GitHub URL'si yerine sabit sürümlü Rowan/hellaswag kullanılır.

In [ ]:
execute(['week12/evaluate.py', '--task', 'prepare', '--limit', str(HELLASWAG_EXAMPLES)])


## 7. Yaklaşık bir saat eğit
İlk `VAL` ve `SAMPLE` rastgele ağırlıklarla ölçülür. İlk compile adımı uzun sürebilir. Her 10 denemede loss/norm/LR/token/s, her 250 başarılı güncellemede val ve aynı prompt'tan örnek yazdırılır. Hücreyi tekrar çalıştırmak son checkpoint'ten devam eder; aynı kaynak/veri ve ayarları koruyun.

In [ ]:
arguments = [
    'week12/train.py', '--data-dir', str(DATA), '--output-dir', str(RESULTS),
    '--device', 'cuda', '--minutes', str(MINUTES),
    '--batch-size', str(BATCH_SIZE), '--sequence-length', str(SEQUENCE_LENGTH),
    '--accumulation-steps', str(ACCUMULATION), '--warmup-steps', '100',
    '--eval-every', '250', '--checkpoint-every', '1000', '--val-batches', '32',
]
if COMPILE:
    arguments += ['--compile']
if BACKUP is not None:
    arguments += ['--backup-dir', str(BACKUP)]
# Rerunning this cell resumes the last saved optimizer/loader/RNG state.
checkpoint = RESULTS / 'last.pt'
if not checkpoint.exists() and BACKUP is not None and (BACKUP / 'last.pt').exists():
    shutil.copyfile(BACKUP / 'last.pt', checkpoint)
if checkpoint.exists():
    arguments += ['--resume', str(checkpoint)]
execute(arguments)


## 8. Kendi modelini HellaSwag'de ölç
İlk 1.000 örnek; ending-only ortalama NLL ana ölçüttür. Her 50 örnekte doğru sayısı yazdırılır. Checkpoint'teki optimizer hali bu hücrede kullanılmaz.

In [ ]:
execute([
    'week12/evaluate.py', '--task', 'own', '--limit', str(HELLASWAG_EXAMPLES),
    '--output-dir', str(RESULTS), '--device', 'cuda',
])


## 9. OpenAI GPT-2'yi aynı örneklerde ölç
Herkese açık, sabit sürümlü GPT-2 ağırlıkları indirilir ve kendi hafta 10 modelimize yüklenir. Aynı tokenizer, aynı örnekler, aynı FP32 puanlama; model API anahtarı gerekmez.

In [ ]:
execute([
    'week12/evaluate.py', '--task', 'openai', '--limit', str(HELLASWAG_EXAMPLES),
    '--output-dir', str(RESULTS), '--device', 'cuda',
])


## 10. Train/val grafiği, örnekler ve karşılaştırma

In [ ]:
from IPython.display import display, Markdown, Image
execute(['week12/report.py', '--output-dir', str(RESULTS)])
display(Markdown((RESULTS / 'submission_report.md').read_text(encoding='utf-8')))
display(Image(filename=str(RESULTS / 'training_curves.png')))


## 11. Sonuç ZIP'ini indir
Metrikler, grafik, örnekler ve kaynak hash'leri ZIP'e eklenir. Checkpoint, veri ve model önbelleği bu küçük ZIP'e eklenmez. Checkpoint Drive yedeğini kontrol edin; Drive kullanmadıysanız ayrıca `files.download(str(RESULTS / 'last.pt'))` ile indirin.

In [ ]:
from google.colab import files
EXPORT = ROOT / 'week12/export' / RUN_NAME
EXPORT.mkdir(parents=True, exist_ok=True)
expected = ['training.json', 'data_manifest.json', 'hellaswag_own.json',
            'hellaswag_openai.json', 'training_curves.png', 'submission_report.md']
for filename in expected:
    source = RESULTS / filename
    assert source.exists(), f'Missing result: {filename}'
    shutil.copyfile(source, EXPORT / filename)
training_result = json.loads((RESULTS / 'training.json').read_text())
assert training_result['status'] == 'complete' and not training_result['smoke_only']
own_result = json.loads((RESULTS / 'hellaswag_own.json').read_text())
openai_result = json.loads((RESULTS / 'hellaswag_openai.json').read_text())
assert own_result['status'] == openai_result['status'] == 'complete'
assert own_result['processed'] == openai_result['processed'] == HELLASWAG_EXAMPLES
assert own_result['selection_sha256'] == openai_result['selection_sha256']
(EXPORT / 'source_manifest.json').write_text(json.dumps(SOURCE_HASHES, indent=2))
archive = shutil.make_archive(str(ROOT / ('week12_' + RUN_NAME + '_results')), 'zip', EXPORT)
if BACKUP is not None:
    shutil.copyfile(archive, BACKUP / Path(archive).name)
files.download(archive)
print('Evidence ZIP:', archive)
print('The checkpoint is separate:', RESULTS / 'last.pt')
print('Drive checkpoint backup:', BACKUP / 'last.pt' if BACKUP is not None else 'not enabled')


## Yorumlama

İngilizce HellaSwag için İngilizce FineWeb-Edu kullanıyoruz. Bir saatlik eğitimin
OpenAI'ın daha uzun ön eğitimini yakalamasını beklemek zorunda değiliz; düşük
sonuç da geçerli bir ölçümdür. Çalıştırılmayan bir aşama tamamlandı sayılmaz.
Ek Türkçe eğitim bu koşuda yoktur. İngilizce HellaSwag Türkçe kalitesini tek başına ölçmez.

- [Karpathy 3:10:21 sonrası](https://www.youtube.com/watch?v=l8pRSuU81PU&t=11421s)
- [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu), ODC-By-1.0
- [HellaSwag](https://huggingface.co/datasets/Rowan/hellaswag)
- [HellaSwag makalesi](https://arxiv.org/abs/1905.07830)
